<a href="https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/dev/models/deep_learning/tft/baseline_tft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Sales Forecasting — TFT baseline

This notebook trains the first Temporal Fusion Transformer baseline for the Walmart weekly sales task.

Main choices:
- validation is the last 39 weeks of `train.csv`, matching the Kaggle test horizon;
- evaluation metric is Kaggle-style WMAE on original sales scale;
- model uses historical sales, Store/Dept identity, holiday flag, and simple known calendar features;
- no economic/external covariates are used in this baseline yet;
- W&B logs config, training curves, validation WMAE, prediction tables, plots, checkpoints, and artifacts;
- no MLflow is used.

Run this on Colab. This notebook is written but not trained locally.

## Install dependencies

`pytorch-forecasting` provides the TFT implementation. If Colab asks to restart after install, restart runtime and run from the imports cell again.

In [ ]:
%pip install -q "torch>=2.3,<3" "pytorch-forecasting>=1.2,<2" "lightning>=2.2,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2"

## Imports and Drive mount

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb

try:
    from lightning.pytorch import Trainer, seed_everything
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from lightning.pytorch.loggers import WandbLogger
except Exception:
    from pytorch_lightning import Trainer, seed_everything
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from pytorch_lightning.loggers import WandbLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

pd.set_option('display.max_columns', 100)
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## Configuration

The baseline is intentionally moderate. TFT is heavier than DLinear, so start with a small hidden size and short training budget. Increase later only after this baseline runs correctly.

In [ ]:
SEED = 42
seed_everything(SEED, workers=True)

CONFIG = {
    "seed": SEED,
    "validation_weeks": 39,
    "encoder_weeks": 52,
    "holiday_weight": 5.0,
    "batch_size": 128,
    "max_epochs": 25,
    "patience": 6,
    "learning_rate": 3e-4,
    "hidden_size": 16,
    "attention_head_size": 2,
    "dropout": 0.10,
    "hidden_continuous_size": 8,
    "gradient_clip_val": 0.1,
    "num_workers": 0,
    "limit_train_batches": 1.0,
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_entity": "kende23-n-a",
    "wandb_group": "tft-baseline",
    "run_name": "tft_baseline_39w",
    "artifact_name": "tft-baseline-39w",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

DATA_DIR = Path('/content/drive/MyDrive/walmart_competition_data')
OUTPUT_DIR = Path('/content/artifacts/tft_baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

## Load data

In [ ]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])

required_train = {'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'}
required_test = {'Store', 'Dept', 'Date', 'IsHoliday'}
missing_train = required_train.difference(train_raw.columns)
missing_test = required_test.difference(test_raw.columns)
if missing_train or missing_test:
    raise ValueError({'missing_train': sorted(missing_train), 'missing_test': sorted(missing_test)})

train_raw = train_raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test_raw = test_raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print('train', train_raw.shape, train_raw['Date'].min(), train_raw['Date'].max())
print('test ', test_raw.shape, test_raw['Date'].min(), test_raw['Date'].max())
display(train_raw.head())

## Build TFT time-indexed data

TFT needs a long time-series table with:
- `time_idx`: integer week index;
- `group_ids`: Store and Dept;
- static categorical identifiers: Store, Dept;
- known future features: holiday flag and calendar features;
- unknown real target: Weekly_Sales.

This baseline does not use `features.csv` or `stores.csv`; those belong to later TFT experiments.

In [ ]:
all_train_dates = pd.Index(sorted(train_raw['Date'].unique()), name='Date')
test_dates = pd.Index(sorted(test_raw['Date'].unique()), name='Date')
val_dates = all_train_dates[-CONFIG['validation_weeks']:]
fit_dates = all_train_dates[:-CONFIG['validation_weeks']]
split_pos = len(fit_dates)

if len(test_dates) != CONFIG['validation_weeks']:
    raise ValueError(f"Expected test horizon {CONFIG['validation_weeks']}, got {len(test_dates)}")
if len(fit_dates) < CONFIG['encoder_weeks'] + CONFIG['validation_weeks']:
    raise ValueError('Not enough fit history for encoder + decoder windows.')

date_to_idx = {date: idx for idx, date in enumerate(all_train_dates)}

def add_time_features(df: pd.DataFrame, date_to_idx: dict) -> pd.DataFrame:
    out = df.copy()
    out['time_idx'] = out['Date'].map(date_to_idx).astype(int)
    iso_week = out['Date'].dt.isocalendar().week.astype(int)
    month = out['Date'].dt.month.astype(int)
    out['week_sin'] = np.sin(2 * np.pi * iso_week / 52.0)
    out['week_cos'] = np.cos(2 * np.pi * iso_week / 52.0)
    out['month_sin'] = np.sin(2 * np.pi * month / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * month / 12.0)
    out['Store'] = out['Store'].astype(str)
    out['Dept'] = out['Dept'].astype(str)
    out['IsHoliday'] = out['IsHoliday'].astype(int).astype(str)
    out['Weekly_Sales'] = out['Weekly_Sales'].astype(float)
    return out

data = add_time_features(train_raw, date_to_idx)

print({
    'n_rows': len(data),
    'n_series': data[['Store', 'Dept']].drop_duplicates().shape[0],
    'n_dates': len(all_train_dates),
    'fit_range': (str(fit_dates.min().date()), str(fit_dates.max().date())),
    'validation_range': (str(val_dates.min().date()), str(val_dates.max().date())),
    'test_horizon': len(test_dates),
})
display(data.head())

## WMAE helpers and seasonal naive reference

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray, holiday_weight: float = 5.0) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

sales_panel = (
    train_raw.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales', aggfunc='sum')
    .reindex(columns=all_train_dates)
    .fillna(0.0)
    .sort_index()
)

holiday_by_date = (
    train_raw[['Date', 'IsHoliday']]
    .drop_duplicates('Date')
    .set_index('Date')
    .reindex(all_train_dates)['IsHoliday']
    .fillna(False)
    .astype(bool)
)

values = sales_panel.to_numpy(dtype=np.float32)
holiday_flags = holiday_by_date.to_numpy(dtype=bool)
actual_val = values[:, split_pos:split_pos + CONFIG['validation_weeks']]
seasonal_naive = values[:, split_pos - 52:split_pos - 52 + CONFIG['validation_weeks']]
val_holidays_matrix = np.tile(holiday_flags[split_pos:split_pos + CONFIG['validation_weeks']], (values.shape[0], 1))
seasonal_naive_wmae = wmae(actual_val.ravel(), seasonal_naive.ravel(), val_holidays_matrix.ravel(), CONFIG['holiday_weight'])

print({
    'n_series': len(sales_panel),
    'validation_series': values.shape[0],
    'seasonal_naive_wmae': seasonal_naive_wmae,
})

## Create TimeSeriesDataSet objects

Training uses only rows before validation. Validation is created from the full table with `predict=True`, so each Store-Dept series gets one direct 39-week forecast.

In [ ]:
training_cutoff = split_pos - 1

training_dataset = TimeSeriesDataSet(
    data[data.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='Weekly_Sales',
    group_ids=['Store', 'Dept'],
    min_encoder_length=CONFIG['encoder_weeks'] // 2,
    max_encoder_length=CONFIG['encoder_weeks'],
    min_prediction_length=CONFIG['validation_weeks'],
    max_prediction_length=CONFIG['validation_weeks'],
    static_categoricals=['Store', 'Dept'],
    time_varying_known_categoricals=['IsHoliday'],
    time_varying_known_reals=['time_idx', 'week_sin', 'week_cos', 'month_sin', 'month_cos'],
    time_varying_unknown_reals=['Weekly_Sales'],
    target_normalizer=GroupNormalizer(groups=['Store', 'Dept'], center=True),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data,
    predict=True,
    stop_randomization=True,
    min_prediction_idx=split_pos,
)

train_loader = training_dataset.to_dataloader(
    train=True,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)
val_loader = validation_dataset.to_dataloader(
    train=False,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)

print({
    'train_batches': len(train_loader),
    'validation_batches': len(val_loader),
    'training_samples': len(training_dataset),
    'validation_samples': len(validation_dataset),
})

## Optional sanity check: PyTorch Forecasting naive baseline

This is separate from our 52-week seasonal naive. It is useful as a library sanity check before training TFT.

In [ ]:
baseline_model = Baseline()
baseline_predictions = baseline_model.predict(val_loader, return_y=True)
print('PyTorch Forecasting Baseline object ran successfully.')

## Start W&B and build TFT

In [ ]:
wandb_logger = WandbLogger(
    project=CONFIG['wandb_project'],
    entity=CONFIG['wandb_entity'],
    group=CONFIG['wandb_group'],
    name=CONFIG['run_name'],
    job_type='baseline_train',
    log_model=False,
    config=CONFIG,
)

run = wandb_logger.experiment
run.config.update({
    'n_series': int(data[['Store', 'Dept']].drop_duplicates().shape[0]),
    'n_train_rows': int(len(data[data.time_idx <= training_cutoff])),
    'n_validation_rows': int(len(data[data.time_idx >= split_pos])),
    'fit_start': str(fit_dates.min().date()),
    'fit_end': str(fit_dates.max().date()),
    'validation_start': str(val_dates.min().date()),
    'validation_end': str(val_dates.max().date()),
    'seasonal_naive_wmae': float(seasonal_naive_wmae),
}, allow_val_change=True)

checkpoint_callback = ModelCheckpoint(
    dirpath=str(OUTPUT_DIR / 'checkpoints'),
    filename='tft-baseline-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
)
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=CONFIG['patience'],
    mode='min',
)
lr_monitor = LearningRateMonitor(logging_interval='epoch')

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=CONFIG['learning_rate'],
    hidden_size=CONFIG['hidden_size'],
    attention_head_size=CONFIG['attention_head_size'],
    dropout=CONFIG['dropout'],
    hidden_continuous_size=CONFIG['hidden_continuous_size'],
    loss=MAE(),
    optimizer='adam',
    log_interval=20,
    reduce_on_plateau_patience=3,
)

print(f'Number of parameters: {tft.size() / 1e3:.1f}k')
run.summary['model_parameters'] = int(tft.size())

## Train

This cell starts training. Do not run it locally in Codex; run it on Colab/GPU.

In [ ]:
trainer = Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    gradient_clip_val=CONFIG['gradient_clip_val'],
    callbacks=[early_stop_callback, checkpoint_callback, lr_monitor],
    logger=wandb_logger,
    enable_checkpointing=True,
    limit_train_batches=CONFIG['limit_train_batches'],
    log_every_n_steps=20,
)

trainer.fit(
    tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

best_checkpoint_path = checkpoint_callback.best_model_path
best_val_loss = float(checkpoint_callback.best_model_score.cpu()) if checkpoint_callback.best_model_score is not None else math.nan
print({'best_checkpoint_path': best_checkpoint_path, 'best_val_loss': best_val_loss})
run.summary['best_checkpoint_path'] = best_checkpoint_path
run.summary['best_val_loss'] = best_val_loss

## Evaluate validation WMAE on original sales scale

We select models by Kaggle-style WMAE, not by TFT's internal normalized validation loss.

In [ ]:
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_checkpoint_path)
best_tft.eval()

prediction = best_tft.predict(
    val_loader,
    mode='prediction',
    return_index=True,
    return_y=True,
    trainer_kwargs={
        'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
        'devices': 1,
    },
)

# pytorch-forecasting returns a Prediction object in recent versions.
pred_values = prediction.output.detach().cpu().numpy() if torch.is_tensor(prediction.output) else np.asarray(prediction.output)
index_df = prediction.index.reset_index(drop=True).copy()

if pred_values.ndim == 3:
    pred_values = pred_values[..., 0]
if pred_values.shape[1] != CONFIG['validation_weeks']:
    raise ValueError(f"Expected prediction horizon {CONFIG['validation_weeks']}, got {pred_values.shape}")

actual_lookup = sales_panel.copy()
records = []
for row_idx, row in index_df.iterrows():
    store = int(row['Store'])
    dept = int(row['Dept'])
    if (store, dept) not in actual_lookup.index:
        continue
    for horizon_idx, date in enumerate(val_dates):
        actual = float(actual_lookup.loc[(store, dept), date])
        pred = float(max(pred_values[row_idx, horizon_idx], 0.0))
        is_holiday = bool(holiday_by_date.loc[date])
        records.append({
            'Store': store,
            'Dept': dept,
            'Date': pd.Timestamp(date),
            'IsHoliday': is_holiday,
            'Weekly_Sales': actual,
            'Prediction': pred,
            'AbsError': abs(actual - pred),
        })

val_pred_df = pd.DataFrame(records)
if val_pred_df.empty:
    raise ValueError('No validation predictions were aligned. Check prediction.index Store/Dept columns.')

validation_wmae = wmae(
    val_pred_df['Weekly_Sales'].to_numpy(),
    val_pred_df['Prediction'].to_numpy(),
    val_pred_df['IsHoliday'].to_numpy(),
    CONFIG['holiday_weight'],
)

improvement_vs_seasonal = 100.0 * (seasonal_naive_wmae - validation_wmae) / seasonal_naive_wmae
print({
    'validation_wmae': validation_wmae,
    'seasonal_naive_wmae': seasonal_naive_wmae,
    'improvement_vs_seasonal_naive_pct': improvement_vs_seasonal,
    'prediction_rows': len(val_pred_df),
})

run.summary['best_validation_wmae'] = float(validation_wmae)
run.summary['seasonal_naive_wmae'] = float(seasonal_naive_wmae)
run.summary['best_improvement_vs_seasonal_naive_pct'] = float(improvement_vs_seasonal)
wandb.log({
    'validation/wmae': float(validation_wmae),
    'validation/seasonal_naive_wmae': float(seasonal_naive_wmae),
    'validation/improvement_vs_seasonal_naive_pct': float(improvement_vs_seasonal),
})

display(val_pred_df.head())

## Save validation predictions, plots, checkpoint metadata, and W&B artifact

In [ ]:
val_pred_path = OUTPUT_DIR / 'tft_baseline_validation_predictions.csv'
val_pred_df.to_csv(val_pred_path, index=False)

summary = {
    'model': 'TemporalFusionTransformer',
    'experiment': 'tft_baseline_39w',
    'best_checkpoint_path': str(best_checkpoint_path),
    'best_val_loss': best_val_loss,
    'best_validation_wmae': float(validation_wmae),
    'seasonal_naive_wmae': float(seasonal_naive_wmae),
    'improvement_vs_seasonal_naive_pct': float(improvement_vs_seasonal),
    'validation_weeks': CONFIG['validation_weeks'],
    'encoder_weeks': CONFIG['encoder_weeks'],
    'n_series': int(data[['Store', 'Dept']].drop_duplicates().shape[0]),
    'train_date_start': str(fit_dates.min().date()),
    'train_date_end': str(fit_dates.max().date()),
    'validation_date_start': str(val_dates.min().date()),
    'validation_date_end': str(val_dates.max().date()),
    'config': CONFIG,
}
summary_path = OUTPUT_DIR / 'tft_baseline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

fig, ax = plt.subplots(figsize=(7, 4))
sample = val_pred_df.sample(min(7000, len(val_pred_df)), random_state=SEED)
ax.scatter(sample['Weekly_Sales'], sample['Prediction'], s=8, alpha=0.25)
max_axis = np.nanpercentile(sample[['Weekly_Sales', 'Prediction']].to_numpy(), 99)
ax.plot([0, max_axis], [0, max_axis], color='red', linewidth=1)
ax.set_title('TFT baseline validation predictions')
ax.set_xlabel('Actual Weekly_Sales')
ax.set_ylabel('Prediction')
plt.tight_layout()
scatter_path = OUTPUT_DIR / 'tft_baseline_validation_scatter.png'
fig.savefig(scatter_path, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(val_pred_df['AbsError'], bins=80)
ax.set_title('TFT baseline validation absolute error distribution')
ax.set_xlabel('Absolute error')
ax.set_ylabel('count')
plt.tight_layout()
error_hist_path = OUTPUT_DIR / 'tft_baseline_abs_error_hist.png'
fig.savefig(error_hist_path, dpi=160)
plt.show()

artifact = wandb.Artifact(CONFIG['artifact_name'], type='model')
artifact.add_file(str(best_checkpoint_path), name='tft_baseline_best.ckpt')
artifact.add_file(str(summary_path))
artifact.add_file(str(val_pred_path))
artifact.add_file(str(scatter_path))
artifact.add_file(str(error_hist_path))
run.log_artifact(artifact, aliases=['baseline', 'latest'])

wandb.log({
    'validation/prediction_table': wandb.Table(dataframe=val_pred_df.sample(min(20000, len(val_pred_df)), random_state=SEED)),
    'validation/scatter': wandb.Image(str(scatter_path)),
    'validation/abs_error_histogram': wandb.Image(str(error_hist_path)),
})

summary

## Optional interpretation helpers

TFT can expose attention/importance plots. These are useful, but they can be slow. Run only after the baseline result is saved.

In [ ]:
RUN_INTERPRETATION = False

if RUN_INTERPRETATION:
    raw_predictions = best_tft.predict(
        val_loader,
        mode='raw',
        return_x=True,
        trainer_kwargs={
            'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
            'devices': 1,
        },
    )
    interpretation = best_tft.interpret_output(raw_predictions.output, reduction='sum')
    figs = best_tft.plot_interpretation(interpretation)
    for name, fig in figs.items():
        fig_path = OUTPUT_DIR / f'tft_interpretation_{name}.png'
        fig.savefig(fig_path, dpi=160, bbox_inches='tight')
        wandb.log({f'interpretation/{name}': wandb.Image(str(fig_path))})
else:
    print('Interpretation skipped. Set RUN_INTERPRETATION=True after baseline training if needed.')

## Finish W&B run

In [ ]:
wandb.finish()